In [ ]:
[{"language":"markdown","newCode":"# Annotated Field Catalog Generator\n\nGenerate an Excel catalog of legacy field names, definitions, origin labels, calculated formulas, and target database field names."},{"language":"python","newCode":"import pandas as pd\nimport openpyxl\nfrom pathlib import Path\n\nsource_excel_path = Path('Coffee Grounds Data.xlsx')\noutput_catalog_path = Path('field_catalog.xlsx')\n\nprint('Source workbook:', source_excel_path)\nprint('Output catalog:', output_catalog_path)"},{"language":"markdown","newCode":"## Load source definitions and metadata from Excel\n\nRead the legacy workbook and inspect the `Collection DB` sheet for the raw field layout and any embedded formulas."},{"language":"python","newCode":"wb = openpyxl.load_workbook(source_excel_path, data_only=False)\nprint('Workbook sheets:', wb.sheetnames)\nsheet_name = 'Collection DB'\nif sheet_name not in wb.sheetnames:\n    raise ValueError(f'Sheet {sheet_name} not found in workbook')\nsheet = wb[sheet_name]  # legacy raw field sheet\nrows = list(sheet.iter_rows(values_only=False))\nheaders = [cell.value for cell in rows[0]]\nprint('Header labels:', headers)\n\nsample_rows = []\nfor row in rows[1:11]:\n    sample_rows.append([cell.value for cell in row[:12]])\nprint('Sample rows (first 10):')\nfor sample in sample_rows:\n    print(sample)"},{"language":"markdown","newCode":"## Parse formulas and identify calculated fields\n\nCapture calculated source fields by detecting Excel formulas in the sheet and storing the expression text."},{"language":"python","newCode":"legacy_fields = []\nformula_columns = set()\n\nfor row in rows[1:]:\n    if all(cell.value is None for cell in row):\n        continue\n    for idx, cell in enumerate(row):\n        if isinstance(cell.value, str) and cell.value.startswith('='):\n            formula_columns.add(idx)\n\nfor idx, label in enumerate(headers):\n    if label is None:\n        continue\n    legacy_fields.append({\n        'legacy_field_name': str(label).strip(),\n        'excel_column_index': idx + 1,\n        'calculated_field': idx in formula_columns,\n        'formula_text': None,\n    })\n\nfor row in rows[1:20]:\n    for item in legacy_fields:\n        idx = item['excel_column_index'] - 1\n        if item['formula_text'] is None and idx < len(row):\n            cell = row[idx]\n            if isinstance(cell.value, str) and cell.value.startswith('='):\n                item['formula_text'] = cell.value\n\nlegacy_df = pd.DataFrame(legacy_fields)\nlegacy_df.head(20)"},{"language":"markdown","newCode":"## Map field origins to alteryx/tableau/excel labels\n\nAssign a source label to each field based on the legacy worksheet and known derived metrics from Alteryx/Tableau."},{"language":"python","newCode":"def infer_target_field_name(label):\n    text = str(label).strip().lower()\n    mapping = {\n        'store number': 'store_number',\n        'store name': 'raw_store_name',\n        'pickup date': 'pickup_date',\n        'week number': 'week_number',\n        'scg mass (lbs)': 'scg_mass_lbs',\n        'net lbs': 'net_lbs',\n        'pickup initiated by': 'pickup_initiated_by',\n        'master gardener': 'master_gardener',\n        'mg deposit date': 'mg_deposit_date',\n        'cardboard (lbs)': 'cardboard_lbs',\n        'food waste (lbs)': 'food_waste_lbs',\n        'route': 'route',\n        'miles driven': 'miles_driven',\n        'truck odometer': 'truck_odometer',\n        'notes': 'notes',\n        'raw days between collections': 'raw_days_between_collections',\n        'raw days since first collection': 'raw_days_since_first_collection',\n    }\n    return mapping.get(text, None)\n\nderived_fields = {\n    'co2e_lbs': 'alteryx',\n    'transportation_co2e': 'tableau',\n    'scg_lbs_per_mile': 'tableau',\n    'co2e_avoided_per_mile': 'tableau',\n    'per_day_lbs': 'tableau',\n    'rubicon_period': 'tableau',\n    'year_and_week': 'tableau',\n    'days_since_first_collection': 'tableau',\n    'days_between_collections': 'tableau',\n    'running_total': 'tableau',\n}\n\ncatalog_rows = []\nfor item in legacy_fields:\n    target = infer_target_field_name(item['legacy_field_name'])\n    origin = 'excel'\n    if item['calculated_field']:\n        origin = 'excel'\n    catalog_rows.append({\n        'legacy_field_name': item['legacy_field_name'],\n        'target_database_field_name': target or 'unknown',\n        'origin_label': origin,\n        'calculated_field': item['calculated_field'],\n        'formula_text': item['formula_text'] or '',\n        'definition': '',\n    })\n\nfor field_name, origin in derived_fields.items():\n    catalog_rows.append({\n        'legacy_field_name': field_name,\n        'target_database_field_name': field_name,\n        'origin_label': origin,\n        'calculated_field': True,\n        'formula_text': '',\n        'definition': '',\n    })\n\ncatalog_df = pd.DataFrame(catalog_rows)\ncatalog_df.head(20)"},{"language":"markdown","newCode":"## Assemble database field definitions DataFrame\n\nBuild the final annotated catalog data frame, including target database field descriptions for each mapped field."},{"language":"python","newCode":"database_description = {\n    'raw_store_name': 'Original store name from Excel source; stored in pickups.raw_store_name',\n    'pickup_date': 'Collection date; stored in pickups.pickup_date',\n    'week_number': 'Week number from legacy source; stored in pickups.week_number',\n    'scg_mass_lbs': 'Raw SCG mass in pounds; stored in pickups.scg_mass_lbs',\n    'net_lbs': 'Net pounds after adjustments; stored in pickups.net_lbs',\n    'pickup_initiated_by': 'Collector name or identifier; stored in pickups.pickup_initiated_by',\n    'master_gardener': 'Master Gardener contribution in pounds; stored in pickups.master_gardener',\n    'mg_deposit_date': 'Deposit date for Master Gardener contributions; stored in pickups.mg_deposit_date',\n    'cardboard_lbs': 'Cardboard weight in pounds; stored in pickups.cardboard_lbs',\n    'food_waste_lbs': 'Food waste weight in pounds; stored in pickups.food_waste_lbs',\n    'route': 'Collection route; stored in pickups.route',\n    'miles_driven': 'Miles driven for collection; stored in pickups.miles_driven',\n    'truck_odometer': 'Truck odometer reading; stored in pickups.truck_odometer',\n    'notes': 'Legacy notes field; stored in pickups.notes',\n    'raw_days_between_collections': 'Raw days between collections from source Excel; stored in pickups.raw_days_between_collections',\n    'raw_days_since_first_collection': 'Raw days since first collection from source Excel; stored in pickups.raw_days_since_first_collection',\n    'co2e_lbs': 'Calculated CO2e in pounds; derived in vw_pickup_base.co2e_lbs',\n    'transportation_co2e': 'Calculated transportation CO2e; derived in vw_pickup_base.transportation_co2e',\n    'scg_lbs_per_mile': 'SCG pounds per mile; derived in vw_pickup_base.scg_lbs_per_mile',\n    'co2e_avoided_per_mile': 'CO2e avoided per mile; derived in vw_pickup_base.co2e_avoided_per_mile',\n    'per_day_lbs': 'Average SCG pounds per day; derived in vw_pickup_base.per_day_lbs',\n    'rubicon_period': 'Rubicon transition label; derived in vw_pickup_base.rubicon_period',\n    'year_and_week': 'Year and ISO week string; derived in vw_pickup_base.year_and_week',\n    'days_since_first_collection': 'Days since first recorded collection; derived in vw_pickup_base.days_since_first_collection',\n    'days_between_collections': 'Days between collections; derived in vw_pickup_base.days_between_collections',\n    'running_total': 'Cumulative net pounds; derived in vw_pickup_base.running_total',\n}\ncatalog_df['database_field_description'] = catalog_df['target_database_field_name'].map(database_description).fillna('Definition unavailable; review mapping manually.')\ncatalog_df"},{"language":"markdown","newCode":"## Export annotated field catalog to Excel\n\nWrite the assembled catalog to `field_catalog.xlsx` with the legacy field name, definition, origin label, calculated field flag, formula text, and target database field name."},{"language":"python","newCode":"catalog_df.to_excel(output_catalog_path, index=False, sheet_name='Field Catalog')\nprint('Wrote annotated field catalog to:', output_catalog_path)"}]